# 03 Feature Engineering

Test feature changes separately from training-policy changes. Report: Section 3.3 and Appendices C, D and F. Feature selection uses training history only.


## 1. Setup
Set the data path, choose a runtime and install the dependencies used below. The `quick` profile checks the workflow with a small budget; use `report` for the report experiments. Place the four original CSV files in one folder.


In [ ]:
from pathlib import Path
# Use the same data directory and run name in all notebooks.
PROFILE = globals().get('PROFILE', 'quick')  # quick: smoke test; report: full training
DATA_SOURCE = globals().get('DATA_SOURCE', 'drive')  # drive, upload or local
DATA_DIR = Path(globals().get('DATA_DIR', '/content/drive/MyDrive/bt-4012-competition-2026'))
OUTPUT_ROOT = Path(globals().get('OUTPUT_ROOT', '/content/drive/MyDrive/BT4012_Final_Report_English'))
RUN_NAME = globals().get('RUN_NAME', 'final_report_english_v1')
INSTALL_DEPENDENCIES = globals().get('INSTALL_DEPENDENCIES', True)
EXPECTED_COMPETITION = globals().get('EXPECTED_COMPETITION', True)
assert PROFILE in ['quick', 'report']
OUT = OUTPUT_ROOT / RUN_NAME / PROFILE


In [ ]:
import sys, subprocess
# Keep the torch/CUDA version supplied by Colab.
PACKAGES = ['numpy>=1.26', 'pandas>=2.0', 'scipy>=1.11', 'scikit-learn>=1.6', 'joblib', 'tabm==0.0.3', 'rtdl-num-embeddings==0.0.12', 'xgboost>=2.0', 'lightgbm>=4.0', 'catboost==1.2.10']
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)


In [ ]:
CSV_FILES = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']

if DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
elif DATA_SOURCE == 'upload':
    from google.colab import files
    DATA_DIR = Path('/content/bt4012_data')
    DATA_DIR.mkdir(exist_ok=True)
    missing = [name for name in CSV_FILES if not (DATA_DIR / name).exists()]
    if missing:
        print('Upload the missing files:', missing)
        for name, content in files.upload().items():
            if name not in missing:
                raise ValueError('Unexpected file: ' + name)
            (DATA_DIR / name).write_bytes(content)
    if str(OUTPUT_ROOT).startswith('/content/drive/'):
        OUTPUT_ROOT = Path('/content/BT4012_Final_Report_English')
        OUT = OUTPUT_ROOT / RUN_NAME / PROFILE
elif DATA_SOURCE != 'local':
    raise ValueError(DATA_SOURCE)
for name in CSV_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUT.mkdir(parents=True, exist_ok=True)
print('Mode:', PROFILE, '| Output:', OUT)


## 2. Required functions
Only the data structures, models and helpers used by this notebook are included below.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from torch.nn import functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from pathlib import Path
from sklearn.preprocessing import QuantileTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
from collections import deque
import gc
import hashlib
import joblib
import json
from torch import nn
import numpy as np
import pandas as pd
import random
from sklearn.metrics import roc_auc_score
import time
import torch
from sklearn.model_selection import train_test_split
import os
torch.set_num_threads(min(4, os.cpu_count() or 1))

VERSION = 'bt4012-colab-report-20261021-v1'

ENGINE_HASH = '3661498eb4e7e545253aeae84b480c6d4c349dc7e125920bf97989a5e96c0840'

FEATURES = [f'feat_{i}' for i in range(1, 166)]

TABM = dict(k=32, n_blocks=2, d_block=512, dropout=0.1, n_bins=48, d_embedding=16, learning_rate=0.002, weight_decay=0.0003)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def serial(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, Path):
        return str(x)
    raise TypeError(type(x).__name__)

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial, allow_nan=False))
    temp.replace(path)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serial).encode()).hexdigest()

def array_hash(x):
    return hashlib.sha256(np.ascontiguousarray(x).tobytes()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 ** 2), b''):
            h.update(block)
    return h.hexdigest()

def seed_all(seed):
    """Set random seeds before constructing and training a model."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def metrics(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    if len(y) != len(p) or not len(y) or (not np.isfinite(p).all()):
        raise ValueError('Empty, nonfinite or unaligned predictions')
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Unknown labels cannot be scored')
    return dict(n=len(y), positives=int(y.sum()), auc=float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None, ap=float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else None)

def profiles(mode):
    """Use a small budget for a smoke test or the report budget for full experiments."""
    if mode not in ['quick', 'report']:
        raise ValueError(mode)
    return dict(mode=mode, seeds=[42] if mode == 'quick' else [17, 42, 73], cutoffs=[21] if mode == 'quick' else [7, 10, 14, 17, 21], max_epochs=2 if mode == 'quick' else 100, patience=1 if mode == 'quick' else 10, trees=30 if mode == 'quick' else None, batch_size=256)

class Dataset:
    """Read the four CSV files and build the transaction graph."""

    def __init__(self, data_dir, expected_competition=True):
        self.data_dir = Path(data_dir)
        self.train = pd.read_csv(self.data_dir / 'train.csv')
        self.test = pd.read_csv(self.data_dir / 'test.csv')
        self.sample = pd.read_csv(self.data_dir / 'sample_submission.csv')
        raw_edges = pd.read_csv(self.data_dir / 'txs_edgelist.csv')
        self.duplicate_edges = int(raw_edges.duplicated().sum())
        self.edges = raw_edges.drop_duplicates(['txId1', 'txId2']).reset_index(drop=True)
        if list(self.train.columns) != ['txId', 'time_step', *FEATURES, 'label']:
            raise ValueError('Unexpected train columns/order')
        if list(self.test.columns) != ['index', 'txId', 'time_step', *FEATURES]:
            raise ValueError('Unexpected test columns/order')
        if list(self.sample.columns) != ['index', 'target']:
            raise ValueError('Submission schema')
        if self.train.txId.duplicated().any() or self.test.txId.duplicated().any():
            raise ValueError('Duplicate IDs')
        if set(self.train.txId) & set(self.test.txId):
            raise ValueError('Overlapping IDs')
        if not np.array_equal(self.sample['index'], self.test['index']):
            raise ValueError('Submission order')
        if not self.test['index'].is_unique:
            raise ValueError('Duplicate submission IDs')
        if not self.train.label.dropna().isin([0, 1]).all():
            raise ValueError('Labels must be 0/1/blank')
        for df in [self.train, self.test]:
            if not np.isfinite(df[FEATURES].to_numpy()).all():
                raise ValueError('Nonfinite raw features')
        if expected_competition:
            assert (len(self.train), len(self.test), len(self.edges)) == (141772, 15329, 234355)
            assert self.train.time_step.between(1, 35).all() and self.test.time_step.between(36, 49).all()
        self.ids = np.unique(np.r_[self.train.txId, self.test.txId, self.edges.to_numpy().ravel()])
        self.src = np.searchsorted(self.ids, self.edges.txId1)
        self.dst = np.searchsorted(self.ids, self.edges.txId2)
        n = len(self.ids)
        self.outgoing = coo_matrix((np.ones(len(self.src)), (self.src, self.dst)), shape=(n, n)).tocsr()
        self.incoming = self.outgoing.T.tocsr()
        self.indeg = np.diff(self.incoming.indptr)
        self.outdeg = np.diff(self.outgoing.indptr)
        self.nc, self.components = connected_components(self.outgoing, directed=False)
        all_rows = pd.concat([self.train, self.test], ignore_index=True).set_index('txId').reindex(self.ids)
        self.node_raw = all_rows[FEATURES].to_numpy(np.float32)
        self.has_features = np.isfinite(self.node_raw).all(1)
        self.node_y = all_rows.label.fillna(-1).to_numpy(np.int64)
        observed = all_rows.time_step.to_numpy(float)
        self.steps = np.full(n, -1, np.int64)
        for c in range(self.nc):
            member = self.components == c
            times = np.unique(observed[member & np.isfinite(observed)])
            if len(times) != 1:
                raise ValueError('Component does not identify exactly one snapshot')
            self.steps[member] = int(times[0])
        if np.any(self.steps[self.src] != self.steps[self.dst]):
            raise ValueError('Cross-snapshot edge')
        q = deque(np.flatnonzero(self.indeg == 0).tolist())
        deg = self.indeg.copy()
        self.order = []
        while q:
            v = q.popleft()
            self.order.append(v)
            for w in self.outgoing.indices[self.outgoing.indptr[v]:self.outgoing.indptr[v + 1]]:
                deg[w] -= 1
                if deg[w] == 0:
                    q.append(int(w))
        if len(self.order) != n:
            raise ValueError('Graph is not a DAG')
        # Unknown labels are graph context, not negative examples.
        self.frame = self.train.loc[self.train.label.notna()].copy().reset_index(drop=True)
        self.frame.label = self.frame.label.astype(np.int64)
        self.pos = np.searchsorted(self.ids, self.frame.txId)
        self.testpos = np.searchsorted(self.ids, self.test.txId)
        self.node_x = np.column_stack([self.node_raw, self.steps, self.indeg, self.outdeg]).astype(np.float32)
        self.raw = self.node_x[self.pos]
        self.y = self.frame.label.to_numpy(np.int64)
        self._extras = {}
        self.data_hashes = {name: file_hash(self.data_dir / name) for name in ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']}
        self.identity = fingerprint(self.data_hashes)

    def split(self, cut, horizon=14):
        t = self.frame.time_step.to_numpy()
        a = np.flatnonzero(t <= cut)
        b = np.flatnonzero((t > cut) & (t <= cut + horizon))
        if not len(a) or not len(b):
            raise ValueError('Empty temporal split')
        return (a, b)

    def neighbor(self, column, direction='up', visibility='targets'):
        key = (column, direction, visibility)
        if key in self._extras:
            return self._extras[key]
        x = self.node_raw[:, FEATURES.index(column)].copy()
        if visibility == 'targets':
            x[(self.node_y < 0) & (self.steps <= 35)] = np.nan
        elif visibility != 'all':
            raise ValueError(visibility)
        receiver, neighbor = (self.dst, self.src) if direction == 'up' else (self.src, self.dst)
        degree = self.indeg if direction == 'up' else self.outdeg
        good = np.isfinite(x[neighbor])
        count = np.bincount(receiver[good], minlength=len(self.ids))
        sums = np.bincount(receiver[good], weights=x[neighbor[good]], minlength=len(self.ids))
        mean = np.divide(sums, count, out=np.full(len(self.ids), np.nan), where=count > 0)
        reason = np.where(count > 0, 0, np.where(degree == 0, 1, 2))
        coverage = np.divide(count, degree, out=np.zeros(len(degree)), where=degree > 0)
        self._extras[key] = (mean, reason, coverage)
        return (mean, reason, coverage)

    def shapes(self):
        if 'shapes' in self._extras:
            return self._extras['shapes']
        n = len(self.ids)
        role = (self.indeg == 1) & (self.outdeg == 2)
        prefix = np.zeros(n)
        suffix = np.zeros(n)
        continuation = np.zeros(n)
        for v in self.order:
            parents = self.incoming.indices[self.incoming.indptr[v]:self.incoming.indptr[v + 1]]
            if role[v]:
                prefix[v] = 1 + prefix[parents].max(initial=0)
        for v in reversed(self.order):
            children = self.outgoing.indices[self.outgoing.indptr[v]:self.outgoing.indptr[v + 1]]
            if len(children):
                continuation[v] = role[children].mean()
            if role[v]:
                suffix[v] = 1 + suffix[children].max(initial=0)
        two = []
        recon = []
        for adj in [self.incoming, self.outgoing]:
            counts = np.zeros(n)
            ratios = np.zeros(n)
            degree = np.diff(adj.indptr)
            for v in range(n):
                first = adj.indices[adj.indptr[v]:adj.indptr[v + 1]]
                terminal = set()
                for w in first:
                    terminal.update(adj.indices[adj.indptr[w]:adj.indptr[w + 1]])
                paths = int(degree[first].sum())
                counts[v] = len(terminal)
                ratios[v] = (paths - len(terminal)) / max(paths, 1)
            two.append(counts)
            recon.append(ratios)
        result = dict(twohop=np.column_stack(two), reconvergence=np.column_stack(recon), branch_chain=np.column_stack([np.where(role, prefix + suffix - 1, 0), continuation]))
        self._extras['shapes'] = result
        return result


In [ ]:
def prepare_num(A, B, scaler='quantile'):
    """Fit the imputer, constant-column mask and scaler using training rows only."""
    # Fit every transformation using the current training fold.
    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    X = imputer.fit_transform(A).astype(np.float32)
    V = imputer.transform(B).astype(np.float32)
    keep = np.ptp(X, axis=0) > 0
    if not keep.any():
        raise ValueError('All numerical inputs constant')
    X = X[:, keep]
    V = V[:, keep]
    if scaler == 'quantile':
        transform = QuantileTransformer(n_quantiles=max(10, min(len(X) // 30, 1000)), output_distribution='normal', subsample=10 ** 9, random_state=0)
        noise = np.random.default_rng(0).normal(0, 1e-05, X.shape).astype(np.float32)
        transform.fit(X + noise)
    elif scaler == 'robust':
        transform = RobustScaler().fit(X)
    elif scaler == 'standard':
        transform = StandardScaler().fit(X)
    elif scaler == 'none':
        transform = None
    else:
        raise ValueError(scaler)
    bundle = (imputer, keep, transform)
    return (apply_num(A, bundle), apply_num(B, bundle), bundle)

def apply_num(X, bundle):
    """Apply a saved numeric transformation without refitting it."""
    imputer, keep, transform = bundle
    X = imputer.transform(X).astype(np.float32)[:, keep]
    return (transform.transform(X) if transform is not None else X).astype(np.float32)

def category_codes(x, levels):
    """Use zero for categories not observed in the training fold."""
    ix = np.searchsorted(levels, x)
    good = ix < len(levels)
    good[good] &= np.asarray(levels)[ix[good]] == x[good]
    return np.where(good, ix + 1, 0).astype(np.int64)

def make_inputs(data, fit, query, variant='raw', scaler='quantile', base='tabm', test=False, genes=None, feature_plan=None):
    """Every fitted quantity is derived ONLY from fit; labels are not neighbor attributes."""
    original_variant = variant
    categorical_extra = variant.startswith('cat3+')
    if categorical_extra:
        variant = variant.split('+', 1)[1]
    A = data.raw[fit].copy()
    B = (data.node_x[data.testpos] if test else data.raw[query]).copy()
    nodes = data.testpos if test else data.pos[query]
    fitnodes = data.pos[fit]
    cols = 168 if base == 'tabm' else 166
    A = A[:, :cols]
    B = B[:, :cols]
    C = np.empty((len(A), 0), np.int64)
    D = np.empty((len(B), 0), np.int64)
    state = dict(variant=original_variant, base=base, fit_ids_hash=array_hash(data.frame.txId.to_numpy()[fit]), scaler=scaler)
    cat = []
    if variant == 'cat3':
        levels = np.unique(A[:, 2])
        C = category_codes(A[:, 2], levels)[:, None]
        D = category_codes(B[:, 2], levels)[:, None]
        A = np.delete(A, 2, axis=1)
        B = np.delete(B, 2, axis=1)
        cat = [len(levels) + 1]
        state['levels'] = levels
    elif variant == 'conditional':
        levels = np.unique(A[:, 2])
        center = {float(x): float(np.median(A[A[:, 2] == x, 52])) if (A[:, 2] == x).sum() >= 20 else float(np.median(A[:, 52])) for x in levels}
        fallback = float(np.median(A[:, 52]))
        state['conditional_centers'] = center
        state['fallback'] = fallback
        a = A[:, 52] - np.array([center.get(float(x), fallback) for x in A[:, 2]])
        b = B[:, 52] - np.array([center.get(float(x), fallback) for x in B[:, 2]])
        A = np.column_stack([A, a])
        B = np.column_stack([B, b])
    elif variant == 'residual':
        xx = A[:, 54] - A[:, 54].mean()
        slope = float(xx @ (A[:, 52] - A[:, 52].mean()) / max(xx @ xx, 1e-12))
        intercept = float(A[:, 52].mean() - slope * A[:, 54].mean())
        state.update(slope=slope, intercept=intercept)
        A = np.column_stack([A, A[:, 52] - intercept - slope * A[:, 54]])
        B = np.column_stack([B, B[:, 52] - intercept - slope * B[:, 54]])
    elif variant == 'degree':
        if base != 'table':
            raise ValueError('TabM already has degree columns; do not duplicate them')
        A = np.column_stack([A, data.node_x[fitnodes, -2:]])
        B = np.column_stack([B, data.node_x[nodes, -2:]])
    elif variant in ['twohop', 'branch_chain', 'reconvergence']:
        z = data.shapes()[variant]
        A = np.column_stack([A, z[fitnodes]])
        B = np.column_stack([B, z[nodes]])
    elif variant == 'roles':
        values = []
        for receiver, neighbor, degree, den in [(data.dst, data.src, data.outdeg, data.indeg), (data.src, data.dst, data.indeg, data.outdeg)]:
            sums = np.bincount(receiver, weights=np.log1p(degree[neighbor]), minlength=len(data.ids))
            values.append(np.divide(sums, den, out=np.zeros(len(den)), where=den > 0))
        z = np.column_stack(values)
        A = np.column_stack([A, z[fitnodes]])
        B = np.column_stack([B, z[nodes]])
    elif variant == 'joint_energy':
        selector = ExtraTreesClassifier(n_estimators=128, min_samples_leaf=8, max_features='sqrt', class_weight='balanced', random_state=0, n_jobs=2).fit(data.raw[fit, :165], data.y[fit])
        iqr = np.subtract(*np.quantile(data.raw[fit, :165], [0.75, 0.25], axis=0))
        eligible = np.flatnonzero(iqr > 1e-06)
        ix = eligible[np.argsort(-selector.feature_importances_[eligible], kind='stable')[:4]]
        center = np.median(data.raw[fit, :165], axis=0)[ix]
        scale = iqr[ix]
        z = np.clip((data.node_raw[:, ix] - center) / scale, -10, 10)
        z[(data.node_y < 0) & (data.steps <= 35)] = np.nan
        good = np.isfinite(z).all(1)
        both = good[data.src] & good[data.dst]
        count = np.bincount(data.dst[good[data.src]], minlength=len(data.ids))
        dif = np.mean((z[data.src[both]] - z[data.dst[both]]) ** 2, axis=1)
        sums = np.bincount(data.dst[both], weights=dif, minlength=len(data.ids))
        energy = np.divide(sums, count, out=np.full(len(data.ids), np.nan), where=count > 0)
        coverage = count / np.maximum(data.indeg, 1)
        extra = np.column_stack([energy, coverage])
        A = np.column_stack([A, extra[fitnodes]])
        B = np.column_stack([B, extra[nodes]])
        state.update(selected_fields=ix, energy_center=center, energy_scale=scale)
    elif variant.startswith('neighbor_'):
        parts = variant.split('_')
        direction, col, kind, policy = (parts[1], f'feat_{parts[2]}', parts[3], '_'.join(parts[4:]))
        mean, reason, coverage = data.neighbor(col, direction)
        value = mean if kind == 'mean' else data.node_raw[:, FEATURES.index(col)] - mean
        if kind == 'abs':
            value = np.abs(value)
        a = value[fitnodes].copy()
        b = value[nodes].copy()
        observed = a[np.isfinite(a)]
        fill = float(np.median(observed)) if len(observed) else 0.0
        if policy == 'zero':
            fill = 0.0
        if policy == 'self':
            a = np.where(np.isfinite(a), a, data.node_raw[fitnodes, FEATURES.index(col)] if kind == 'mean' else 0.0)
            b = np.where(np.isfinite(b), b, data.node_raw[nodes, FEATURES.index(col)] if kind == 'mean' else 0.0)
        elif policy != 'native':
            a = np.where(np.isfinite(a), a, fill)
            b = np.where(np.isfinite(b), b, fill)
        A = np.column_stack([A, a])
        B = np.column_stack([B, b])
        if policy in ['median_flag', 'self']:
            A = np.column_stack([A, reason[fitnodes] if policy == 'self' else ~np.isfinite(value[fitnodes])])
            B = np.column_stack([B, reason[nodes] if policy == 'self' else ~np.isfinite(value[nodes])])
        state.update(neighbor_fill=fill, visibility='targets', reason_codes=['observed', 'no_edges', 'no_visible_features'])
    elif variant == 'snapshot':
        for j in [52, 54]:
            z = pd.Series(data.node_raw[:, j]).groupby(data.steps).rank(pct=True).to_numpy()
            visible = data.has_features & ((data.node_y >= 0) | (data.steps > 35))
            values = np.where(visible, data.node_raw[:, j], np.nan)
            z = pd.Series(values).groupby(data.steps).rank(pct=True).to_numpy()
            A = np.column_stack([A, z[fitnodes]])
            B = np.column_stack([B, z[nodes]])
    elif variant in ['trend_residual', 'snapshot_center']:
        for j in [100, 102]:
            if variant == 'trend_residual':
                med = pd.Series(A[:, j]).groupby(data.frame.time_step.iloc[fit].to_numpy()).median()
                design = np.column_stack([np.ones(len(med)), med.index])
                coef = np.linalg.lstsq(design, med.to_numpy(), rcond=None)[0]
                A[:, j] -= coef[0] + coef[1] * data.steps[fitnodes]
                B[:, j] -= coef[0] + coef[1] * data.steps[nodes]
                state[f'trend_{j}'] = coef
            else:
                z = np.where(data.has_features & ((data.node_y >= 0) | (data.steps > 35)), data.node_raw[:, j], np.nan)
                centers = pd.Series(z).groupby(data.steps).median()
                A[:, j] -= centers.reindex(data.steps[fitnodes]).to_numpy()
                B[:, j] -= centers.reindex(data.steps[nodes]).to_numpy()
    elif variant.startswith('drop'):
        if feature_plan is not None:
            drop = feature_plan['deletions'][variant]
            if drop is None:
                raise FeatureUnavailable('No training-fold redundant pair with |r|>=0.98')
        else:
            selector = ExtraTreesClassifier(n_estimators=128, min_samples_leaf=8, random_state=0, n_jobs=2).fit(A[:, :165], data.y[fit])
            weak = np.argsort(selector.feature_importances_, kind='stable')
            if categorical_extra:
                weak = weak[weak != 2]
            if variant == 'drop_redundant':
                corr = np.abs(np.corrcoef(A[:, :165], rowvar=False))
                np.fill_diagonal(corr, 0)
                if categorical_extra:
                    corr[2, :] = corr[:, 2] = 0
                i, j = np.unravel_index(np.nanargmax(corr), corr.shape)
                drop = [min([i, j], key=lambda k: selector.feature_importances_[k])]
            else:
                drop = weak[:int(variant[-1])].tolist()
        state['cat3_index_after_drop'] = 2 - sum((i < 2 for i in drop))
        A = np.delete(A, drop, axis=1)
        B = np.delete(B, drop, axis=1)
        state['dropped_fields'] = drop
    elif variant.startswith('interaction'):
        if not genes:
            raise ValueError('Train-only expression plan is required')
        for g in genes[:int(variant[-1])]:
            i, j = (g['a'], g['b'])
            op = g['op']
            fn = {'sum': lambda x: x[:, i] + x[:, j], 'difference': lambda x: x[:, i] - x[:, j], 'product': lambda x: x[:, i] * x[:, j], 'absolute_difference': lambda x: np.abs(x[:, i] - x[:, j])}[op]
            A = np.column_stack([A, fn(A)])
            B = np.column_stack([B, fn(B)])
        state['genes'] = genes
    elif variant == 'no_time':
        A = np.delete(A, 165, 1)
        B = np.delete(B, 165, 1)
    elif variant != 'raw':
        raise ValueError(variant)
    if categorical_extra:
        levels = np.unique(data.raw[fit, 2])
        C = category_codes(data.raw[fit, 2], levels)[:, None]
        D = category_codes(data.node_x[nodes, 2], levels)[:, None]
        index = state.get('cat3_index_after_drop', 2)
        A = np.delete(A, index, 1)
        B = np.delete(B, index, 1)
        cat = [len(levels) + 1]
        state['levels'] = levels
    if scaler == 'native':
        X, V = (A.astype(np.float32), B.astype(np.float32))
        prep = None
    else:
        X, V, prep = prepare_num(A, B, scaler)
    state.update(numerical_preprocessor=prep, cardinalities=cat)
    return (X, V, C, D, state)


In [ ]:
def tree_model(name, y, seed, profile):
    """Construct the selected tree or linear baseline with fixed parameters."""
    w = float((y == 0).sum() / max((y == 1).sum(), 1))
    small = profile['trees']
    if name in ['random_forest', 'extra_trees']:
        cls = RandomForestClassifier if name == 'random_forest' else ExtraTreesClassifier
        return cls(n_estimators=small or 500, min_samples_leaf=2, max_features='sqrt', class_weight='balanced', n_jobs=2, random_state=seed)
    if name == 'hist_gradient_boosting':
        return HistGradientBoostingClassifier(max_iter=small or 500, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=30, l2_regularization=1, class_weight='balanced', early_stopping=False, random_state=seed)
    if name == 'xgboost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=small or 1200, learning_rate=0.025, max_depth=6, min_child_weight=5, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1, scale_pos_weight=w, tree_method='hist', n_jobs=2, random_state=seed)
    if name == 'lightgbm':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=small or 600, learning_rate=0.025, num_leaves=31, min_child_samples=40, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1, scale_pos_weight=w, n_jobs=2, random_state=seed, verbosity=-1)
    if name == 'catboost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(iterations=small or 1200, learning_rate=0.025, depth=6, l2_leaf_reg=3, class_weights=[1, w], thread_count=2, random_seed=seed, verbose=False, allow_writing_files=False)
    raise ValueError(name)

def build_net(name, X, cardinalities, config):
    """Construct a neural model from random initial weights."""
    import tabm, rtdl_num_embeddings as emb
    cfg = dict(config)
    cfg.pop('learning_rate')
    cfg.pop('weight_decay')
    bins = emb.compute_bins(torch.from_numpy(X), n_bins=cfg.pop('n_bins'))
    module = emb.PiecewiseLinearEmbeddings(bins, cfg.pop('d_embedding'), activation=False, version='B')
    return tabm.TabM.make(n_num_features=X.shape[1], cat_cardinalities=cardinalities, d_out=2, num_embeddings=module, **cfg).to(DEVICE)

def head_loss(logits, y):
    """Average cross-entropy across all TabM members and training examples."""
    return F.cross_entropy(logits.flatten(0, 1), y.repeat_interleave(logits.shape[1]))

def forward_net(net, name, x, c, pool, y, idx=None):
    return net(x, c if c.shape[1] else None)

@torch.inference_mode()
def net_predict(net, name, V, D, pool, y, batch=256):
    """Run inference and average the 32 member probabilities for TabM."""
    net.eval()
    parts = []
    for start in range(0, len(V), batch):
        x = torch.as_tensor(V[start:start + batch], device=DEVICE)
        c = torch.as_tensor(D[start:start + batch], device=DEVICE)
        logits = forward_net(net, name, x, c, pool, y)
        p = logits.softmax(-1)[..., 1].mean(1)
        parts.append(p.cpu().numpy())
    return np.concatenate(parts)

def fit_net(name, seed, X, C, y, V, D, profile, config=None, val_y=None, epochs=None):
    """Select an epoch count on inner validation, or refit for a fixed number of epochs."""
    seed_all(seed)
    cfg = dict(TABM if config is None else config)
    net = build_net(name, X, [] if C.shape[1] == 0 else [int(max(C.max(), D.max(initial=0))) + 1], cfg)
    tx = torch.as_tensor(X, device=DEVICE)
    tc = torch.as_tensor(C, device=DEVICE)
    ty = torch.as_tensor(y, device=DEVICE)
    lr = cfg['learning_rate']
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=cfg['weight_decay'])
    curve = []
    best = -np.inf
    selected = 0
    stale = 0
    for epoch in range(1, (epochs or profile['max_epochs']) + 1):
        net.train()
        loss_sum = 0.0
        for idx in torch.randperm(len(tx), device=DEVICE).split(profile['batch_size']):
            opt.zero_grad(set_to_none=True)
            logits = forward_net(net, name, tx[idx], tc[idx], tx, ty, idx)
            loss = head_loss(logits, ty[idx])
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            loss_sum += float(loss.detach()) * len(idx)
        row = dict(epoch=epoch, loss=loss_sum / len(tx))
        if epochs is None:
            if val_y is None:
                raise ValueError('Selection needs inner labels, never outer labels')
            row.update(metrics(val_y, net_predict(net, name, V, D, tx, ty)))
            if row['auc'] is None:
                raise ValueError('Inner selection window has one class')
            if row['auc'] > best:
                best = row['auc']
                selected = epoch
                stale = 0
            else:
                stale += 1
        else:
            selected = epoch
        curve.append(row)
        if epochs is None and stale >= profile['patience']:
            break
    p = net_predict(net, name, V, D, tx, ty)
    checkpoint = dict(state_dict={k: v.detach().cpu().clone() for k, v in net.state_dict().items()}, config=cfg, name=name, input_dim=X.shape[1], cardinalities=[] if C.shape[1] == 0 else [int(C.max()) + 1], epochs=selected, seed=seed, initialization='random', pretrained=False)
    del net, opt, tx, tc, ty
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (p, selected, curve, checkpoint)

def predict_tree(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    return model.predict(X)

def split_indices(data, cut, horizon=14, kind='time', split_seed=20261016, inner_horizon=3):
    """Keep the epoch-selection split inside the outer training history."""
    if kind == 'time':
        a, b = data.split(cut, horizon)
        innercut = cut - min(inner_horizon, cut - 4)
        ia, ib = data.split(innercut, cut - innercut)
    elif kind == 'random':
        a, b = train_test_split(np.arange(len(data.frame)), test_size=0.3, stratify=data.y, random_state=split_seed)
        ia, ib = train_test_split(a, test_size=0.3, stratify=data.y[a], random_state=split_seed + 1)
    elif kind == 'component':
        groups = data.components[data.pos]
        ix = np.arange(len(data.frame))
        a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=split_seed).split(ix, groups=groups))
        x, z = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=split_seed + 1).split(a, groups=groups[a]))
        ia, ib = (a[x], a[z])
    else:
        raise ValueError(kind)
    for z in [a, b, ia, ib]:
        if len(np.unique(data.y[z])) < 2:
            raise ValueError('Split needs both labels')
    assert not np.intersect1d(a, b).size and set(ia) | set(ib) <= set(a)
    return tuple((np.sort(z) for z in [a, b, ia, ib]))

def run_trial(data, out, model='tabm', variant='raw', cut=21, horizon=14, seed=42, profile=None, config=None, inner_horizon=3, base='tabm', kind='time', split_seed=20261016, scaler=None, stage='experiments', genes=None, epoch_policy='own', control_variant='raw', feature_plan=None):
    """Fit one historical trial without using outer validation labels for epoch selection."""
    profile = profile or profiles('quick')
    config = dict(TABM if config is None else config)
    neural = model in ['tabm']
    scaler = scaler or ('quantile' if neural else 'none')
    spec = dict(version=VERSION, data=data.identity, stage=stage, model=model, variant=variant, cut=cut, horizon=horizon, seed=seed, profile=profile, config=config, inner_horizon=inner_horizon, base=base, kind=kind, split_seed=split_seed, scaler=scaler, genes=genes, epoch_policy=epoch_policy, control_variant=control_variant, feature_plan=feature_plan, engine_hash=globals().get('ENGINE_HASH', VERSION))
    key = fingerprint(spec)
    job = Path(out) / stage / f'{model}_{variant}_{key[:16]}'
    if (job / 'complete.json').exists():
        record = json.loads((job / 'complete.json').read_text())
        assert record['spec'] == json.loads(json.dumps(spec))
        p = pd.read_csv(job / 'predictions.csv', float_precision='round_trip')
        assert array_hash(p.txId.to_numpy()) == record['query_ids_hash']
        assert metrics(p.label, p.prediction) == record['metrics']
        print('REUSE', model, variant, cut, seed, flush=True)
        return record
    a, b, ia, ib = split_indices(data, cut, horizon, kind, split_seed, inner_horizon)
    job.mkdir(parents=True, exist_ok=True)
    save_json(job / 'started.json', spec)
    start = time.time()
    X, V, C, D, bundle = make_inputs(data, a, b, variant, scaler, base, genes=genes, feature_plan=feature_plan)
    if neural:
        selection_variant = control_variant if epoch_policy == 'control' else variant
        A, B, E, G, inner = make_inputs(data, ia, ib, selection_variant, scaler, base, genes=genes, feature_plan=feature_plan)
        _, ep, selection, _ = fit_net(model, seed, A, E, data.y[ia], B, G, profile, config, val_y=data.y[ib])
        p, _, curve, weights = fit_net(model, seed, X, C, data.y[a], V, D, profile, config, epochs=ep)
        torch.save(weights, job / 'model.pt')
        joblib.dump(dict(outer=bundle, inner=inner), job / 'preprocessing.joblib')
        save_json(job / 'selection.json', selection)
        save_json(job / 'refit_curve.json', curve)
    else:
        ep = None
        estimator = tree_model(model, data.y[a], seed, profile)
        if C.shape[1]:
            levels = bundle['cardinalities'][0]
            X = np.column_stack([X, np.eye(levels)[C[:, 0]]])
            V = np.column_stack([V, np.eye(levels)[D[:, 0]]])
        estimator.fit(X, data.y[a])
        p = predict_tree(estimator, V)
        joblib.dump(dict(estimator=estimator, preprocessor=bundle), job / 'model.joblib')
    pred = data.frame.iloc[b][['txId', 'time_step', 'label']].copy()
    pred['prediction'] = np.asarray(p, dtype=np.float64)
    pred.to_csv(job / 'predictions.csv', index=False)
    record = dict(spec=spec, metrics=metrics(data.y[b], p), selected_epochs=ep, train_rows=len(a), train_ids_hash=array_hash(data.frame.txId.to_numpy()[a]), query_ids_hash=array_hash(pred.txId.to_numpy()), prediction_hash=file_hash(job / 'predictions.csv'), seconds=time.time() - start, path=str(job), outer_labels_used_for_epoch_selection=False, complete=True)
    save_json(job / 'complete.json', record)
    print('DONE', model, variant, cut, seed, record['metrics'], flush=True)
    return record


In [ ]:
def prediction_records(out, stage=None):
    records = []
    for f in Path(out).rglob('complete.json'):
        r = json.loads(f.read_text())
        if all((k in r for k in ['spec', 'path', 'metrics', 'query_ids_hash'])) and 'stage' in r['spec'] and (r['spec'].get('engine_hash') == globals().get('ENGINE_HASH', VERSION)) and (stage is None or r['spec']['stage'] == stage):
            records.append(r)
    return records

def aggregate_runs(records):
    """Average seed predictions before calculating each window metric."""
    groups = {}
    for r in records:
        s = r['spec']
        g = {k: v for k, v in s.items() if k != 'seed'}
        key = fingerprint(g)
        groups.setdefault(key, (g, []))[1].append(r)
    rows = []
    for spec, rs in groups.values():
        rs = sorted(rs, key=lambda r: r['spec']['seed'])
        seeds = [r['spec']['seed'] for r in rs]
        if len(seeds) != len(set(seeds)):
            raise ValueError('Duplicate seeded trials')
        expected = spec['profile']['seeds']
        complete = sorted(seeds) == sorted(expected)
        reference = None
        parts = []
        for r in rs:
            frame = pd.read_csv(Path(r['path']) / 'predictions.csv', float_precision='round_trip')
            if reference is None:
                reference = frame
            else:
                np.testing.assert_array_equal(reference[['txId', 'time_step', 'label']], frame[['txId', 'time_step', 'label']])
            parts.append(frame.prediction.to_numpy())
        bag = np.column_stack(parts).mean(1)
        row = dict(stage=spec['stage'], model=spec['model'], variant=spec['variant'], cut=spec['cut'], horizon=spec['horizon'], kind=spec['kind'], config_id=fingerprint(spec['config'])[:10], inner_horizon=spec['inner_horizon'], scaler=spec['scaler'], base=spec['base'], complete=complete, seeds=seeds, **metrics(reference.label, bag))
        rows.append(row)
    return pd.DataFrame(rows)

def paired_table(table, control='raw'):
    """Compare each feature variant with its matching control."""
    t = table.loc[table.complete].copy()
    keys = ['stage', 'model', 'cut', 'horizon', 'kind', 'config_id', 'inner_horizon', 'scaler', 'base']
    ref = t.loc[t.variant == control, keys + ['auc', 'ap']].rename(columns={'auc': 'control_auc', 'ap': 'control_ap'})
    result = t.merge(ref, on=keys, validate='many_to_one')
    result['delta_auc'] = result.auc - result.control_auc
    result['delta_ap'] = result.ap - result.control_ap
    return result

def search_expressions(data, cut, profile):
    """Bounded train-only screening; outer future labels are never accessed."""
    a, b = data.split(cut - 6, 3)
    X = data.raw[a, :165]
    V = data.raw[b, :165]
    proxy = tree_model('lightgbm', data.y[a], 42, profile)
    proxy.fit(X, data.y[a])
    top = np.argsort(-proxy.feature_importances_)[:8]
    rng = np.random.default_rng(20261013)
    seen = {}
    population = []
    operators = ['sum', 'difference', 'product', 'absolute_difference']
    size = 4 if profile['mode'] == 'quick' else 12
    generations = 1 if profile['mode'] == 'quick' else 3
    for generation in range(generations):
        while len(population) < size:
            i, j = map(int, rng.choice(top, 2, replace=False))
            op = str(rng.choice(operators))
            gene = dict(a=i, b=j, op=op)
            if gene not in population:
                population.append(gene)
        for gene in population:
            key = json.dumps(gene, sort_keys=True)
            if key in seen:
                continue
            fn = {'sum': lambda z: z[:, gene['a']] + z[:, gene['b']], 'difference': lambda z: z[:, gene['a']] - z[:, gene['b']], 'product': lambda z: z[:, gene['a']] * z[:, gene['b']], 'absolute_difference': lambda z: np.abs(z[:, gene['a']] - z[:, gene['b']])}[gene['op']]
            m = tree_model('lightgbm', data.y[a], 42, profile).fit(np.column_stack([X, fn(X)]), data.y[a])
            score = metrics(data.y[b], m.predict_proba(np.column_stack([V, fn(V)]))[:, 1])['auc']
            seen[key] = (score, gene)
        ranked = sorted(seen.values(), key=lambda z: -z[0])
        population = [g for _, g in ranked[:2]]
    return ([g for _, g in sorted(seen.values(), key=lambda z: -z[0])[:2]], dict(evaluations=len(seen), search_train_end=cut - 6, search_valid_end=cut - 3, outer_labels_used=False))

def adjacency_lists(src, dst, n):
    outgoing = coo_matrix((np.ones(len(src)), (src, dst)), shape=(n, n)).tocsr()
    incoming = outgoing.transpose().tocsr()
    return (incoming, outgoing)

def degree_features(src, dst, n):
    inc = np.bincount(dst, minlength=n).astype(np.float64)
    out = np.bincount(src, minlength=n).astype(np.float64)
    total = inc + out
    values = {'degree_in': inc, 'degree_out': out, 'degree_total': total, 'degree_log_in': np.log1p(inc), 'degree_log_out': np.log1p(out), 'degree_log_total': np.log1p(total), 'degree_collection_ratio': (inc + 1) / (out + 1), 'degree_imbalance': (out - inc) / (total + 1), 'degree_source': inc == 0, 'degree_sink': out == 0, 'degree_one_in_one_out': (inc == 1) & (out == 1), 'degree_many_in_few_out': (inc >= 2) & (out <= 1), 'degree_few_in_many_out': (inc <= 1) & (out >= 2), 'degree_many_in_many_out': (inc >= 2) & (out >= 2)}
    return values

def neighbor_statistics(values, src, dst, n, upstream):
    receiver, neighbor = (dst, src) if upstream else (src, dst)
    available = np.isfinite(values[neighbor])
    rows = receiver[available]
    x = values[neighbor[available]].astype(np.float64)
    count = np.bincount(rows, minlength=n)
    mean = np.full(n, np.nan)
    std = np.full(n, np.nan)
    maximum = np.full(n, -np.inf)
    has = count > 0
    total = np.bincount(rows, weights=x, minlength=n)
    total_sq = np.bincount(rows, weights=x * x, minlength=n)
    mean[has] = total[has] / count[has]
    std[has] = np.sqrt(np.maximum(total_sq[has] / count[has] - mean[has] ** 2, 0))
    np.maximum.at(maximum, rows, x)
    maximum[~has] = np.nan
    q = pd.DataFrame({'receiver': rows, 'value': x}).groupby('receiver').value.quantile([0.25, 0.5, 0.75]).unstack()
    quantiles = np.full((n, 3), np.nan)
    if len(q):
        quantiles[q.index.to_numpy()] = q.to_numpy()
    return {'count': count, 'mean': mean, 'std': std, 'max': maximum, 'q25': quantiles[:, 0], 'median': quantiles[:, 1], 'q75': quantiles[:, 2]}

def topology_features(src, dst, n):
    incoming, outgoing = adjacency_lists(src, dst, n)
    in_degree = np.diff(incoming.indptr)
    out_degree = np.diff(outgoing.indptr)
    remaining = in_degree.copy()
    queue = deque(np.flatnonzero(remaining == 0).tolist())
    order = []
    root_depth = np.zeros(n, dtype=np.int64)
    while queue:
        node = queue.popleft()
        order.append(node)
        children = outgoing.indices[outgoing.indptr[node]:outgoing.indptr[node + 1]]
        for child in children:
            root_depth[child] = max(root_depth[child], root_depth[node] + 1)
            remaining[child] -= 1
            if remaining[child] == 0:
                queue.append(int(child))
    if len(order) != n:
        raise ValueError('Graph is not a DAG')
    leaf_depth = np.zeros(n, dtype=np.int64)
    is_chain = (in_degree == 1) & (out_degree == 1)
    prefix, suffix = (np.zeros(n, dtype=np.int64), np.zeros(n, dtype=np.int64))
    for node in order:
        if is_chain[node]:
            parent = incoming.indices[incoming.indptr[node]]
            prefix[node] = prefix[parent] + 1 if is_chain[parent] else 1
    for node in reversed(order):
        children = outgoing.indices[outgoing.indptr[node]:outgoing.indptr[node + 1]]
        if len(children):
            leaf_depth[node] = leaf_depth[children].max() + 1
        if is_chain[node]:
            child = children[0]
            suffix[node] = suffix[child] + 1 if is_chain[child] else 1
    up2, down2 = (np.zeros(n), np.zeros(n))
    siblings, coparents = (np.zeros(n), np.zeros(n))
    roles = {f'topology_{direction}_{role}_neighbors': np.zeros(n) for direction in ['up', 'down'] for role in ['collection', 'distribution']}
    for node in range(n):
        parents = incoming.indices[incoming.indptr[node]:incoming.indptr[node + 1]]
        children = outgoing.indices[outgoing.indptr[node]:outgoing.indptr[node + 1]]
        ancestors, descendants, shared_parent, shared_child = (set(), set(), set(), set())
        for parent in parents:
            ancestors.update(incoming.indices[incoming.indptr[parent]:incoming.indptr[parent + 1]])
            shared_parent.update(outgoing.indices[outgoing.indptr[parent]:outgoing.indptr[parent + 1]])
        for child in children:
            descendants.update(outgoing.indices[outgoing.indptr[child]:outgoing.indptr[child + 1]])
            shared_child.update(incoming.indices[incoming.indptr[child]:incoming.indptr[child + 1]])
        shared_parent.discard(node)
        shared_child.discard(node)
        up2[node], down2[node] = (len(ancestors), len(descendants))
        siblings[node], coparents[node] = (len(shared_parent), len(shared_child))
        for direction, adjacent in [('up', parents), ('down', children)]:
            roles[f'topology_{direction}_collection_neighbors'][node] = np.sum((in_degree[adjacent] >= 2) & (out_degree[adjacent] <= 1))
            roles[f'topology_{direction}_distribution_neighbors'][node] = np.sum((in_degree[adjacent] <= 1) & (out_degree[adjacent] >= 2))
    return {'topology_up2_unique': up2, 'topology_down2_unique': down2, 'topology_shared_parent_peers': siblings, 'topology_shared_child_peers': coparents, 'topology_up_expansion': up2 / (in_degree + 1), 'topology_down_expansion': down2 / (out_degree + 1), 'topology_root_longest_depth': root_depth, 'topology_leaf_longest_depth': leaf_depth, 'topology_relative_depth': root_depth / (root_depth + leaf_depth + 1), 'topology_chain_prefix': prefix, 'topology_chain_suffix': suffix, 'topology_chain_length': np.where(is_chain, prefix + suffix - 1, 0), **roles}


In [ ]:
def snapshot_statistics(values, steps, epsilon):
    series = pd.Series(values)
    groups = series.groupby(steps)
    rank = groups.rank(method='average', pct=True).to_numpy()
    median = groups.transform('median').to_numpy()
    residual = values - median
    mad = pd.Series(np.abs(residual)).groupby(steps).transform('median').to_numpy()
    robust_z = np.clip(residual / np.maximum(1.4826 * mad, epsilon), -20, 20)
    return (rank, residual, robust_z)

class GraphFeatureData:

    def __init__(self, root: Path):
        self.train = pd.read_csv(root / 'train.csv')
        test = pd.read_csv(root / 'test.csv')
        edges = pd.read_csv(root / 'txs_edgelist.csv').drop_duplicates()
        known = pd.concat([self.train, test], ignore_index=True).set_index('txId')
        if not known.index.is_unique:
            raise ValueError('Duplicate feature-row identifiers')
        self.ids = np.union1d(edges.txId1, edges.txId2)
        if not known.index.isin(self.ids).all():
            raise ValueError('Missing feature-table transaction in graph')
        self.src = np.searchsorted(self.ids, edges.txId1.to_numpy())
        self.dst = np.searchsorted(self.ids, edges.txId2.to_numpy())
        n = len(self.ids)
        adjacency = coo_matrix((np.ones(len(edges)), (self.src, self.dst)), shape=(n, n))
        nc, components = connected_components(adjacency, directed=True, connection='weak')
        ordered = known.reindex(self.ids)
        observed = ordered.time_step.to_numpy(dtype=np.float64)
        component_steps = np.full(nc, -1, dtype=np.int64)
        for c in range(nc):
            times = np.unique(observed[(components == c) & np.isfinite(observed)])
            if len(times) != 1:
                raise ValueError('Component crosses time steps or has no time metadata')
            component_steps[c] = int(times[0])
        self.steps = component_steps[components]
        if not np.array_equal(self.steps[self.src], self.steps[self.dst]):
            raise ValueError('Cross-time-step edges are not permitted')
        self.features = [c for c in self.train.columns if c.startswith('feat_')]
        self.values = ordered[self.features].to_numpy(dtype=np.float32)
        self.y = ordered.label.fillna(-1).to_numpy(dtype=np.int64)
        if not np.isin(self.y, [-1, 0, 1]).all():
            raise ValueError('Unexpected labels')
        self.has_features = np.isfinite(observed)
        self.base = np.column_stack([self.values, self.steps]).astype(np.float32)
        self.base_names = self.features + ['time_step']
        self.degree = degree_features(self.src, self.dst, n)
        self.topology = None
        self.selected = {}
        self.stats_cache = {}
        self.neighbor_structure = {}
        self.metadata = {'nodes': n, 'edges': len(edges), 'components': int(nc), 'missing_feature_nodes': int((~self.has_features).sum())}

    def select_features(self, cutoff, count=8):
        if cutoff not in self.selected:
            fit = (self.y >= 0) & (self.steps <= cutoff)
            X = self.values[fit]
            if not np.isfinite(X).all():
                raise ValueError('Selector expects finite original training features')
            selector = ExtraTreesClassifier(n_estimators=128, min_samples_leaf=10, class_weight='balanced', n_jobs=8, random_state=42)
            selector.fit(X, self.y[fit])
            indices = np.argsort(-selector.feature_importances_, kind='stable')[:count]
            self.selected[cutoff] = {'indices': indices.tolist(), 'names': [self.features[i] for i in indices], 'train_end': cutoff, 'training_rows': int(fit.sum()), 'selector': selector.get_params(), 'importance': selector.feature_importances_.tolist()}
        return self.selected[cutoff]

    def epsilon(self, values, cutoff):
        fit = self.has_features & (self.steps <= cutoff) & np.isfinite(values)
        return max(0.001, 0.01 * float(np.std(values[fit])))

    def visible_values(self, cutoff):
        available = self.has_features.copy()
        unavailable_context = (self.steps > cutoff) & (self.steps <= 35) & (self.y < 0)
        available[unavailable_context] = False
        values = self.values.copy()
        values[~available] = np.nan
        return (values, available)

    def neighbor_stats(self, index, upstream, cutoff, values):
        key = (cutoff, index, upstream)
        if key not in self.stats_cache:
            self.stats_cache[key] = neighbor_statistics(values[:, index], self.src, self.dst, len(self.ids), upstream)
        return self.stats_cache[key]

    def make_features(self, group, cutoff):
        if group not in GROUPS:
            raise ValueError(group)
        extras = {}
        selected = None
        n = len(self.ids)
        if group == 'degree':
            extras = self.degree
        elif group == 'topology':
            if self.topology is None:
                self.topology = topology_features(self.src, self.dst, n)
            extras = self.topology
        elif group in {'neighbors', 'snapshot'}:
            selected = self.select_features(cutoff)
            visible, available_features = self.visible_values(cutoff)
            if group == 'neighbors':
                for j in selected['indices']:
                    values = visible[:, j]
                    name = self.features[j]
                    eps = self.epsilon(values, cutoff)
                    up = self.neighbor_stats(j, True, cutoff, visible)
                    down = self.neighbor_stats(j, False, cutoff, visible)
                    for direction, stats in [('up', up), ('down', down)]:
                        for statistic in ['mean', 'median', 'std', 'q25', 'q75', 'max']:
                            extras[f'neighbor_{direction}_{name}_{statistic}'] = stats[statistic]
                        extras[f'neighbor_{direction}_{name}_self_minus_mean'] = values - stats['mean']
                        extras[f'neighbor_{direction}_{name}_robust_z'] = np.clip((values - stats['median']) / np.maximum(stats['q75'] - stats['q25'], eps), -20, 20)
                    extras[f'neighbor_{name}_down_minus_up'] = down['mean'] - up['mean']
                if cutoff not in self.neighbor_structure:
                    structure = {}
                    for direction, upstream, degree_name in [('up', True, 'degree_in'), ('down', False, 'degree_out')]:
                        receiver, neighbor = (self.dst, self.src) if upstream else (self.src, self.dst)
                        available = np.bincount(receiver, weights=available_features[neighbor], minlength=n)
                        structure[f'neighbor_{direction}_available_count'] = available
                        structure[f'neighbor_{direction}_coverage'] = available / np.maximum(self.degree[degree_name], 1)
                        for label, degree in [('in_degree', self.degree['degree_in']), ('out_degree', self.degree['degree_out'])]:
                            stats = neighbor_statistics(degree, self.src, self.dst, n, upstream)
                            for statistic in ['mean', 'max', 'std']:
                                structure[f'neighbor_{direction}_{label}_{statistic}'] = stats[statistic]
                    self.neighbor_structure[cutoff] = structure
                extras.update(self.neighbor_structure[cutoff])
            else:
                sources = [(self.features[j], visible[:, j]) for j in selected['indices']]
                sources += [(name, self.degree[name]) for name in ['degree_in', 'degree_out', 'degree_total']]
                for name, values in sources:
                    rank, residual, z = snapshot_statistics(values, self.steps, self.epsilon(values, cutoff))
                    extras[f'snapshot_{name}_percentile'] = rank
                    extras[f'snapshot_{name}_median_residual'] = residual
                    extras[f'snapshot_{name}_mad_z'] = z
        extra_names = list(extras)
        matrix = self.base if not extras else np.column_stack([self.base] + [extras[name] for name in extra_names]).astype(np.float32)
        if np.isinf(matrix).any():
            raise ValueError('Infinite engineered feature')
        return (matrix, self.base_names + extra_names, selected)


In [ ]:
class SmallGraphFeatureData(GraphFeatureData):

    def make_features(self, group, cutoff):
        if group == 'raw':
            return (self.base, self.base_names, None)
        selected = None
        if group == 'degree_pair':
            extras = {'graph_in_degree': self.degree['degree_in'], 'graph_out_degree': self.degree['degree_out']}
        elif group == 'two_hop_pair':
            if self.topology is None:
                self.topology = topology_features(self.src, self.dst, len(self.ids))
            extras = {name: self.topology[name] for name in ['topology_up2_unique', 'topology_down2_unique']}
        elif group in {'neighbor_pair', 'snapshot_pair'}:
            selected = self.select_features(cutoff, count=1)
            index = selected['indices'][0]
            feature = self.features[index]
            visible, _ = self.visible_values(cutoff)
            if group == 'neighbor_pair':
                up = self.neighbor_stats(index, True, cutoff, visible)
                down = self.neighbor_stats(index, False, cutoff, visible)
                extras = {f'neighbor_up_{feature}_self_minus_mean': visible[:, index] - up['mean'], f'neighbor_down_{feature}_self_minus_mean': visible[:, index] - down['mean']}
            else:
                rank, _, _ = snapshot_statistics(visible[:, index], self.steps, 0.001)
                degree_rank, _, _ = snapshot_statistics(self.degree['degree_total'], self.steps, 0.001)
                extras = {f'snapshot_{feature}_percentile': rank, 'snapshot_total_degree_percentile': degree_rank}
        else:
            raise ValueError(group)
        matrix = np.column_stack([self.base] + list(extras.values())).astype(np.float32)
        if matrix.shape[1] != 168 or np.isinf(matrix).any():
            raise ValueError('Small ablations must add exactly two finite-or-missing columns')
        return (matrix, self.base_names + list(extras), selected)

class FeatureUnavailable(Exception):
    """A data-dependent hypothesis lacks its required training-fold support."""

def safe_scale(x):
    center = np.nanmedian(x, axis=0)
    scale = 1.4826 * np.nanmedian(np.abs(x - center), axis=0)
    fallback = np.nanstd(x, axis=0)
    scale = np.where(scale > 1e-06, scale, np.maximum(fallback, 1e-06))
    return (center, scale)

def campaign_category_codes(train, values):
    """Exact float levels are tokens, never int-truncated standardized values."""
    levels = np.unique(train[np.isfinite(train)])
    code = np.searchsorted(levels, values)
    valid = np.isfinite(values) & (code < len(levels))
    positions = np.flatnonzero(valid)
    valid[positions] &= levels[code[positions]] == values[positions]
    return (np.where(valid, code, -1).astype(np.int32), levels)

class CampaignFeatures(GraphFeatureData):

    def __init__(self, root):
        super().__init__(root)
        self.index = {name: i for i, name in enumerate(self.features)}
        self.matrix_cache = {}

    def fit_mask(self, cutoff):
        return (self.y >= 0) & (self.steps <= cutoff)

    def audit_and_plan(self, output):
        cutoff = 14
        fit = self.fit_mask(cutoff)
        frame = pd.DataFrame(self.values[fit], columns=self.features)
        selection = self.select_features(cutoff, count=165)
        importance = dict(zip(self.features, selection['importance']))
        pearson, spearman = (frame.corr(), frame.corr(method='spearman'))
        duplicate_pairs, affine_pairs, correlated = ([], [], [])
        for i, a in enumerate(self.features):
            for b in self.features[i + 1:]:
                if np.array_equal(frame[a].to_numpy(), frame[b].to_numpy(), equal_nan=True):
                    duplicate_pairs.append([a, b])
                elif abs(pearson.loc[a, b]) > 1 - 1e-07:
                    affine_pairs.append([a, b, float(pearson.loc[a, b])])
                if abs(spearman.loc[a, b]) > 0.97 and min(importance[a], importance[b]) > 0.001:
                    correlated.append({'a': a, 'b': b, 'rho': float(spearman.loc[a, b]), 'importance_min': min(importance[a], importance[b])})
        correlated.sort(key=lambda r: (-r['importance_min'], -abs(r['rho']), r['a'], r['b']))
        chosen_pairs = [['feat_53', 'feat_55']]
        for row in correlated:
            pair = [row['a'], row['b']]
            if set(pair) != {'feat_53', 'feat_55'} and (not any((set(pair) == set(p) for p in duplicate_pairs))):
                chosen_pairs.append(pair)
            if len(chosen_pairs) >= 4:
                break
        weak = sorted(self.features, key=lambda c: (importance[c], c))[:6]
        strongest = sorted(self.features, key=lambda c: (-importance[c], c))[:8]
        specs = [{'id': 'raw', 'kind': 'raw', 'columns': []}]

        def add(identity, kind, columns, **options):
            specs.append({'id': identity, 'kind': kind, 'columns': columns, **options})
        for columns in [['feat_3'], ['feat_113'], ['feat_3', 'feat_113']]:
            add('native_cat_' + '_'.join((c[5:] for c in columns)), 'native_category', columns)
        add('drop_15', 'drop', ['feat_15'])
        add('rare_state_15', 'rare_state', ['feat_15'])
        add('category_frequency_3_113', 'category_frequency', ['feat_3', 'feat_113'])
        add('category_joint_3_113', 'category_joint', ['feat_3', 'feat_113'])
        for category in ['feat_3', 'feat_113']:
            add('conditional_53_' + category[5:], 'conditional', [category, 'feat_53'])
        add('residual_binned_53_55', 'binned_residual', ['feat_53', 'feat_55'], bins=20)
        for pair in chosen_pairs:
            suffix = '_'.join((c[5:] for c in pair))
            add('residual_linear_' + suffix, 'linear_residual', pair)
            drop = min(pair, key=lambda c: (importance[c], c))
            add('drop_correlated_' + suffix, 'drop', [drop])
        add('product_53_55', 'product', ['feat_53', 'feat_55'])
        add('graph_residual_matched_53', 'graph_residual', ['feat_53'])
        add('graph_state_agreement_3', 'graph_agreement', ['feat_3'])
        add('top8_row_anomaly', 'row_anomaly', strongest)
        add('signed_log_53_55', 'signed_log', ['feat_53', 'feat_55'])
        add('drop_time_step', 'drop', ['time_step'])
        for column in weak:
            add('drop_weak_' + column[5:], 'drop', [column])
        for i in range(0, len(weak), 2):
            add('drop_weak_pair_' + '_'.join((c[5:] for c in weak[i:i + 2])), 'drop', weak[i:i + 2])
        for i, pair in enumerate(duplicate_pairs[:2]):
            add(f'drop_exact_duplicate_{i}', 'drop', [pair[1]], exact_duplicate_of=pair[0])
        for i, (a, b, _) in enumerate(affine_pairs[:4]):
            drop = min([a, b], key=lambda c: (importance[c], c))
            add(f'drop_affine_candidate_{i}', 'drop', [drop], affine_candidate_of=a if drop == b else b)
        deduplicated, seen = ([], set())
        for spec in specs:
            signature = (spec['kind'], tuple(spec['columns']))
            if signature not in seen:
                deduplicated.append(spec)
                seen.add(signature)
        audit = {'hypothesis_selection_cutoff': cutoff, 'hypothesis_selection_rows': int(fit.sum()), 'unique_counts': frame.nunique().to_dict(), 'mode_shares': {c: float(frame[c].value_counts(normalize=True).iloc[0]) for c in self.features}, 'exact_duplicate_columns': duplicate_pairs, 'affine_duplicate_candidates': affine_pairs, 'high_spearman_pairs': correlated, 'fixed_relation_pairs': chosen_pairs, 'weak_feature_candidates': weak, 'strong_feature_candidates': strongest, 'feature_importance': importance, 'duplicate_feature_rows': int(frame.duplicated().sum()), 'notes': ['Duplicate feature vectors do not justify deleting distinct transactions.', 'Low cardinality or uniform spacing does not establish nominal category meaning.', 'Low impurity importance is only a deletion hypothesis, not proof of uselessness.', 'Neither test features nor outer-period class labels choose hypothesis identities.']}
        output.mkdir(parents=True, exist_ok=True)
        (output / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
        pearson.to_csv(output / 'training14_pearson.csv')
        spearman.to_csv(output / 'training14_spearman.csv')
        return deduplicated

    def transform(self, spec, cutoff):
        identity = (spec['id'], cutoff)
        if identity in self.matrix_cache:
            return self.matrix_cache[identity]
        fit = self.fit_mask(cutoff)
        columns = spec['columns']
        values = [self.values[:, self.index[c]].astype(np.float64) for c in columns if c in self.index]
        kind = spec['kind']
        evidence = {'statistic_fit_cutoff': cutoff, 'statistic_fit_rows': int(fit.sum()), 'kind': kind}
        base, names, categories, extras = (self.base, self.base_names.copy(), [], [])
        if kind == 'raw':
            pass
        elif kind == 'drop':
            keep = [i for i, c in enumerate(names) if c not in columns]
            if 'exact_duplicate_of' in spec:
                a, b = (self.index[columns[0]], self.index[spec['exact_duplicate_of']])
                if not np.array_equal(self.values[fit, a], self.values[fit, b], equal_nan=True):
                    raise FeatureUnavailable('Exact duplicate deletion failed its training-fold recheck')
            if 'affine_candidate_of' in spec:
                a, b = (self.index[columns[0]], self.index[spec['affine_candidate_of']])
                correlation = float(np.corrcoef(self.values[fit, a], self.values[fit, b])[0, 1])
                if not np.isfinite(correlation) or abs(correlation) < 1 - 1e-07:
                    raise FeatureUnavailable('Near-affine deletion failed its training-fold correlation recheck')
                evidence['training_pair_pearson'] = correlation
            base, names = (base[:, keep], [names[i] for i in keep])
        elif kind == 'native_category':
            categories = columns.copy()
        elif kind == 'rare_state':
            x = values[0]
            level, counts = np.unique(x[fit], return_counts=True)
            mode = level[np.argmax(counts)]
            extras = [(x != mode).astype(float), np.sign(x - mode) * np.log1p(np.abs(x - mode))]
            evidence['training_mode'] = float(mode)
        elif kind == 'category_frequency':
            for x in values:
                code, levels = campaign_category_codes(x[fit], x)
                count = np.bincount(code[fit], minlength=len(levels))
                frequency = np.zeros(len(x))
                seen = code >= 0
                frequency[seen] = count[code[seen]] / fit.sum()
                extras.append(frequency)
        elif kind == 'category_joint':
            a, al = campaign_category_codes(values[0][fit], values[0])
            b, bl = campaign_category_codes(values[1][fit], values[1])
            table = np.zeros((len(al), len(bl)))
            np.add.at(table, (a[fit], b[fit]), 1)
            joint = (table + 1) / (fit.sum() + table.size)
            pa, pb = (joint.sum(axis=1), joint.sum(axis=0))
            seen = (a >= 0) & (b >= 0)
            frequency, lift = (np.zeros(len(a)), np.zeros(len(a)))
            frequency[seen] = joint[a[seen], b[seen]]
            lift[seen] = np.log(joint[a[seen], b[seen]] / (pa[a[seen]] * pb[b[seen]]))
            extras = [frequency, lift]
        elif kind == 'conditional':
            category, x = values
            code, levels = campaign_category_codes(category[fit], category)
            center, scale = safe_scale(x[fit])
            medians, scales = (np.full(len(levels), center), np.full(len(levels), scale))
            for k in range(len(levels)):
                part = x[fit & (code == k)]
                if len(part) >= 20:
                    medians[k], scales[k] = safe_scale(part)
            reference, denominator = (np.full(len(x), center), np.full(len(x), scale))
            seen = code >= 0
            reference[seen], denominator[seen] = (medians[code[seen]], scales[code[seen]])
            residual = x - reference
            extras = [residual, np.clip(residual / denominator, -30, 30)]
            evidence['category_levels'] = levels.tolist()
        elif kind in {'linear_residual', 'binned_residual'}:
            target, predictor = values
            if kind == 'linear_residual':
                centered = predictor[fit] - predictor[fit].mean()
                slope = float(centered @ (target[fit] - target[fit].mean()) / max(centered @ centered, 1e-12))
                intercept = float(target[fit].mean() - slope * predictor[fit].mean())
                expected = intercept + slope * predictor
                evidence.update(slope=slope, intercept=intercept)
            else:
                edges = np.unique(np.quantile(predictor[fit], np.linspace(0, 1, spec['bins'] + 1)))[1:-1]
                bucket = np.searchsorted(edges, predictor)
                medians = np.array([np.median(target[fit & (bucket == k)]) if (fit & (bucket == k)).any() else np.median(target[fit]) for k in range(len(edges) + 1)])
                expected = medians[bucket]
                evidence.update(bin_edges=edges.tolist(), bin_medians=medians.tolist())
            residual = target - expected
            extras = [residual, np.abs(residual)]
        elif kind == 'product':
            z = []
            for x in values:
                center, scale = safe_scale(x[fit])
                z.append(np.clip((x - center) / scale, -30, 30))
            extras = [z[0] * z[1], np.abs(z[0] - z[1])]
        elif kind in {'graph_residual', 'graph_agreement'}:
            x = values[0].copy()
            available = self.has_features & ((self.y >= 0) | (self.steps > 35))
            x[~available] = np.nan
            evidence['graph_visibility'] = 'target numerical rows only in BOTH training and evaluation; all edges retained'
            for upstream in [True, False]:
                if kind == 'graph_residual':
                    stats = neighbor_statistics(x, self.src, self.dst, len(self.ids), upstream)
                    extras.append(x - stats['mean'])
                else:
                    receiver, neighbor = (self.dst, self.src) if upstream else (self.src, self.dst)
                    valid = np.isfinite(x[receiver]) & np.isfinite(x[neighbor])
                    count = np.bincount(receiver[valid], minlength=len(x))
                    equal = np.bincount(receiver[valid], weights=x[receiver[valid]] == x[neighbor[valid]], minlength=len(x))
                    extras.append(np.divide(equal, count, out=np.full(len(x), np.nan), where=count > 0))
        elif kind == 'row_anomaly':
            subset = np.column_stack(values)
            center, scale = safe_scale(subset[fit])
            z = np.clip((subset - center) / scale, -30, 30)
            extras = [np.sqrt(np.mean(z ** 2, axis=1)), np.max(np.abs(z), axis=1)]
        elif kind == 'signed_log':
            base = base.copy()
            for column, x in zip(columns, values):
                center, scale = safe_scale(x[fit])
                z = (x - center) / scale
                base[:, names.index(column)] = np.sign(z) * np.log1p(np.abs(z))
        else:
            raise ValueError(kind)
        if extras:
            if len(extras) > 2:
                raise ValueError('At most two added features per hypothesis')
            names += [f"{spec['id']}_derived_{i + 1}" for i in range(len(extras))]
            base = np.column_stack([base] + extras).astype(np.float32)
        if np.isinf(base).any():
            raise ValueError('Nonfinite feature transformation')
        result = (base, names, categories, evidence)
        return result

    def prepared(self, spec, cutoff, valid_end, cache, test_ids=None):
        token = hashlib.sha256(json.dumps([spec, cutoff, valid_end, bool(test_ids is not None)], sort_keys=True).encode()).hexdigest()[:24]
        path, meta_path = (cache / (token + '.npz'), cache / (token + '.json'))
        if path.exists() and meta_path.exists():
            return (path, json.loads(meta_path.read_text()))
        matrix, names, categories, evidence = self.transform(spec, cutoff)
        fit = self.fit_mask(cutoff)
        valid = (self.y >= 0) & (self.steps > cutoff) & (self.steps <= valid_end)
        positions = np.flatnonzero(valid) if test_ids is None else np.searchsorted(self.ids, test_ids)
        X, V = (matrix[fit].copy(), matrix[positions].copy())
        categorical_indices = [names.index(c) for c in categories]
        numeric_indices = [i for i in range(len(names)) if i not in categorical_indices]
        imputer = SimpleImputer(strategy='median', keep_empty_features=True)
        X[:, numeric_indices] = imputer.fit_transform(X[:, numeric_indices])
        V[:, numeric_indices] = imputer.transform(V[:, numeric_indices])
        level_counts = []
        for i in categorical_indices:
            combined = np.concatenate([X[:, i], V[:, i]])
            codes, levels = campaign_category_codes(X[:, i], combined)
            X[:, i], V[:, i] = (codes[:len(X)], codes[len(X):])
            level_counts.append(len(levels))
        if not np.isfinite(X).all() or not np.isfinite(V).all():
            raise ValueError('Imputation did not produce finite matrices')
        feature_hash = hashlib.sha256(X.tobytes() + V.tobytes()).hexdigest()
        metadata = {'feature_names': names, 'categorical_indices': categorical_indices, 'category_level_counts': level_counts, 'features_hash': feature_hash, 'train_end': cutoff, 'valid_end': valid_end, 'train_rows': int(fit.sum()), 'valid_rows': len(positions), 'evidence': evidence, 'imputer_statistics': imputer.statistics_.tolist()}
        cache.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, X=X, V=V, y=self.y[fit], valid_y=self.y[positions], ids=self.ids[positions], steps=self.steps[positions])
        meta_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        return (path, metadata)

GROUPS = ['raw', 'degree', 'neighbors', 'topology', 'snapshot']


In [ ]:
def native_tree_inputs(name, estimator, X, V, indices, level_counts):
    if not indices:
        return (X, V)
    X = pd.DataFrame(X)
    V = pd.DataFrame(V)
    if name in ['xgboost', 'lightgbm']:
        for i, n in zip(indices, level_counts):
            levels = list(range(-1, n))
            X[i] = pd.Categorical(X[i].astype(int), categories=levels)
            V[i] = pd.Categorical(V[i].astype(int), categories=levels)
        if name == 'xgboost':
            estimator.set_params(enable_categorical=True)
    elif name == 'catboost':
        for i in indices:
            X[i] = X[i].astype(int).astype(str)
            V[i] = V[i].astype(int).astype(str)
        estimator.set_params(cat_features=indices)
    elif name == 'hist_gradient_boosting':
        X, V = (X.to_numpy(copy=True), V.to_numpy(copy=True))
        for i in indices:
            X[X[:, i] < 0, i] = np.nan
            V[V[:, i] < 0, i] = np.nan
        estimator.set_params(categorical_features=indices)
    else:
        raise ValueError('Native categorical unsupported by ' + name)
    return (X, V)

def run_original_tree_features(data, out, store, group, cut, seed, model, profile, campaign=False, overrides=None):
    """Run source graph-pair/full-group or original bounded campaign features, without historical result dependencies."""
    spec = dict(version=VERSION, data=data.identity, stage='original_campaign' if campaign else 'original_graph_pairs', model=model, variant=group['id'] if campaign else group, cut=cut, horizon=7, seed=seed, profile=profile, config=overrides or {}, inner_horizon=3, base='table', kind='time', scaler='native-category/train-median', engine_hash=globals().get('ENGINE_HASH', VERSION))
    job = Path(out) / spec['stage'] / f'{model}_{fingerprint(spec)[:16]}'
    job.mkdir(parents=True, exist_ok=True)
    if (job / 'complete.json').exists():
        return json.loads((job / 'complete.json').read_text())
    if campaign:
        cache = Path(out) / 'feature_matrices' / f'campaign_{data.identity[:8]}'
        path, meta = store.prepared(group, cut, cut + 7, cache)
        z = np.load(path)
        X, V, y, vy, ids, steps = [z[k] for k in ['X', 'V', 'y', 'valid_y', 'ids', 'steps']]
        categorical = meta['categorical_indices']
        counts = meta['category_level_counts']
    else:
        matrix, names, selected = store.make_features(group, cut)
        a = (store.y >= 0) & (store.steps <= cut)
        b = (store.y >= 0) & (store.steps > cut) & (store.steps <= cut + 7)
        imputer = SimpleImputer(strategy='median', keep_empty_features=True)
        X = imputer.fit_transform(matrix[a])
        V = imputer.transform(matrix[b])
        y = store.y[a]
        vy = store.y[b]
        ids = store.ids[b]
        steps = store.steps[b]
        categorical = []
        counts = []
        meta = dict(feature_names=names, selection=selected, imputer=imputer, visibility='original source policy')
    estimator = tree_model(model, y, seed, profile)
    params = dict(overrides or {})
    mode = params.pop('_weight_mode', None)
    if mode == 'unweighted':
        params['class_weights'] = [1.0, 1.0]
    estimator.set_params(**params)
    X, V = native_tree_inputs(model, estimator, X, V, categorical, counts)
    estimator.fit(X, y)
    p = predict_tree(estimator, V)
    joblib.dump(dict(model=estimator, metadata=meta), job / 'model.joblib')
    pred = pd.DataFrame(dict(txId=ids, time_step=steps, label=vy, prediction=p.astype(float)))
    pred.to_csv(job / 'predictions.csv', index=False)
    r = dict(spec=spec, metrics=metrics(vy, p), query_ids_hash=array_hash(ids), prediction_hash=file_hash(job / 'predictions.csv'), path=str(job), complete=True, preprocessing_scope_end=cut)
    save_json(job / 'complete.json', r)
    return r

def source_tree_grids():
    from itertools import product
    grids = {m: [{}] for m in ['xgboost', 'lightgbm', 'catboost', 'hist_gradient_boosting', 'extra_trees']}
    for depth, child, reg, schedule in product([4, 6, 8], [3, 10], [1.0, 5.0], [(0.025, 1200), (0.05, 600)]):
        grids['xgboost'].append(dict(max_depth=depth, min_child_weight=child, reg_lambda=reg, learning_rate=schedule[0], n_estimators=schedule[1]))
    for leaves, child, reg, schedule in product([15, 31, 63], [20, 60], [1.0, 5.0], [(0.025, 600), (0.05, 300)]):
        grids['lightgbm'].append(dict(num_leaves=leaves, min_child_samples=child, reg_lambda=reg, learning_rate=schedule[0], n_estimators=schedule[1]))
    for depth, reg, schedule, weighted in product([4, 6, 8], [3.0, 10.0], [(0.025, 1200), (0.05, 600)], [True, False]):
        c = dict(depth=depth, l2_leaf_reg=reg, learning_rate=schedule[0], iterations=schedule[1])
        if not weighted:
            c['_weight_mode'] = 'unweighted'
        grids['catboost'].append(c)
    for leaves, child, reg in product([15, 31, 63], [20, 60], [0.1, 1.0, 10.0]):
        grids['hist_gradient_boosting'].append(dict(max_leaf_nodes=leaves, min_samples_leaf=child, l2_regularization=reg))
    for features, child, depth in product(['sqrt', 0.3, 0.7], [1, 2, 5], [None, 12, 24]):
        grids['extra_trees'].append(dict(max_features=features, min_samples_leaf=child, max_depth=depth))
    return grids

def tabm_shap_plan(data, cut, profile):
    """Original LightGBM TreeSHAP proxy, evaluated ONLY inside outer training."""
    from lightgbm import LGBMClassifier
    fit, query = data.split(cut - 6, 3)
    X, V, C, D, bundle = make_inputs(data, fit, query, 'cat3')
    names = np.delete(np.arange(168), 2)[bundle['numerical_preprocessor'][1]]
    importance = []
    # Each seed starts from newly initialized weights.
    for seed in profile['seeds']:
        model = LGBMClassifier(n_estimators=30 if profile['mode'] == 'quick' else 180, learning_rate=0.04, num_leaves=31, min_child_samples=30, colsample_bytree=0.9, subsample=0.9, subsample_freq=1, reg_lambda=1.0, verbosity=-1, n_jobs=2, random_state=seed)
        A = np.column_stack([X, C])
        B = np.column_stack([V, D])
        model.fit(A, data.y[fit], categorical_feature=[X.shape[1]])
        shap = model.booster_.predict(B, pred_contrib=True)
        margin = model.booster_.predict(B, raw_score=True)
        np.testing.assert_allclose(shap.sum(1), margin, atol=1e-07, rtol=1e-07)
        importance.append(np.abs(shap[:, :-1]).mean(0)[:X.shape[1]])
    importance = np.mean(importance, axis=0)
    candidates = [i for i, n in enumerate(names) if n < 165]
    ranked = sorted(candidates, key=lambda i: (importance[i], names[i]))
    weak = [int(names[i]) for i in ranked[:6]]
    corr = np.corrcoef(X[:, candidates], rowvar=False)
    pairs = []
    for a, i in enumerate(candidates):
        for b, j in enumerate(candidates[a + 1:], a + 1):
            if np.isfinite(corr[a, b]) and abs(corr[a, b]) >= 0.98:
                remove = min([i, j], key=lambda k: (importance[k], -names[k]))
                pairs.append((float(importance[remove]), -abs(float(corr[a, b])), int(names[remove])))
    pairs.sort()
    return dict(cutoff=cut, shap_train_end=cut - 6, shap_query_end=cut - 3, deletions=dict(drop1=weak[:1], drop2=weak[:2], drop_redundant=[pairs[0][2]] if pairs else None), shap_importance={str(int(n)): float(v) for n, v in zip(names, importance)}, important_fields=[int(names[i]) for i in reversed(ranked[-8:])], outer_labels_used=False)


In [ ]:
P = profiles(PROFILE)


## 3. Load and check the data


In [ ]:
# Labelled rows are the only rows used by supervised models.
data = Dataset(DATA_DIR, expected_competition=EXPECTED_COMPETITION)
print('Labelled training rows:', len(data.frame), '| Unknown labels:', int(data.train.label.isna().sum()), '| Test rows:', len(data.test))


## 4. Tree-model graph features
Compare degree, two-hop reach, a downstream feature difference and snapshot ranks with the original input. The snapshot example uses feat_53 and feat_55; it is a compact reconstruction of the original larger search.


In [ ]:
FE_TREE_MODELS = globals().get('FE_TREE_MODELS', ['lightgbm'] if PROFILE == 'quick' else ['xgboost','lightgbm','catboost','hist_gradient_boosting','random_forest'])
TREE_VARIANTS = ['raw','degree'] if PROFILE == 'quick' else ['raw','degree','twohop','neighbor_down_106_signed_median','snapshot']
for cutoff in ([21] if PROFILE == 'quick' else [14,21]):
    for model in FE_TREE_MODELS:
        for variant in TREE_VARIANTS:
            for seed in P['seeds']:
                run_trial(data, OUT, model=model, variant=variant, cut=cutoff, horizon=7,
                    seed=seed, profile=P, base='table', stage='tree_features')
display(paired_table(aggregate_runs(prediction_records(OUT, 'tree_features'))))


## 5. TabM feature representation
Compare categorical feat_3, conditional deviations and residuals. Use the control model to select epochs so input changes are evaluated separately.


In [ ]:
TABM_FEATURE_VARIANTS = globals().get('TABM_FEATURE_VARIANTS', ['raw','cat3'] if PROFILE == 'quick' else ['raw','cat3','conditional','residual'])
for cutoff in P['cutoffs']:
    for variant in TABM_FEATURE_VARIANTS:
        for seed in P['seeds']:
            run_trial(data, OUT, variant=variant, cut=cutoff, seed=seed, profile=P,
                      stage='tabm_features', epoch_policy='control')
display(paired_table(aggregate_runs(prediction_records(OUT, 'tabm_features'))))


## 6. Graph shape and upstream deviation
Compare branching/chain shape, reconvergence and joint upstream deviation with neighbour coverage. No neighbouring labels are used as features.


In [ ]:
RUN_GRAPH_FEATURES = globals().get('RUN_GRAPH_FEATURES', False)
if RUN_GRAPH_FEATURES:
    for cutoff in P['cutoffs']:
        for variant in ['raw','branch_chain','reconvergence','joint_energy']:
            for seed in P['seeds']:
                run_trial(data, OUT, variant=variant, cut=cutoff, seed=seed, profile=P,
                    stage='graph_features', epoch_policy='control')
    display(paired_table(aggregate_runs(prediction_records(OUT, 'graph_features'))))
else:
    print('Optional graph-feature study skipped.')


## 7. Missing neighbour attributes
Graph-only nodes have no supplied attributes. Compare native missing values, median/zero replacement, indicators and self fallback. This is an optional 75-combination study before repeats across windows and seeds.


In [ ]:
RUN_MISSING_STUDY = globals().get('RUN_MISSING_STUDY', False)
if RUN_MISSING_STUDY:
    for cutoff in [14,21]:
        for model in ['xgboost','lightgbm','catboost','hist_gradient_boosting','extra_trees']:
            for seed in P['seeds']:
                run_trial(data, OUT, model=model, cut=cutoff, horizon=7, seed=seed,
                    profile=P, base='table', scaler='native', stage='neighbor_missing')
                for column, kind in [(130,'mean'),(142,'mean'),(106,'abs')]:
                    for policy in ['native','median','zero','median_flag','self']:
                        run_trial(data, OUT, model=model, variant=f'neighbor_down_{column}_{kind}_{policy}',
                            cut=cutoff, horizon=7, seed=seed, profile=P, base='table',
                            scaler='native', stage='neighbor_missing')
    display(paired_table(aggregate_runs(prediction_records(OUT, 'neighbor_missing'))))
else:
    print('Optional missing-attribute study skipped.')


## 8. Feature deletion and interactions
Use an inner LightGBM SHAP proxy to propose deletions and a bounded training-only search to propose interactions. This reconstruction evaluates expressions on raw features before numeric preprocessing.


In [ ]:
RUN_ADVANCED_FE = globals().get('RUN_ADVANCED_FE', False)
if RUN_ADVANCED_FE:
    for cutoff in ([21] if PROFILE == 'quick' else [14,21]):
        genes, search = search_expressions(data, cutoff, P)
        plan = tabm_shap_plan(data, cutoff, P)
        save_json(OUT / f'feature_plan_cut{cutoff}.json', dict(genes=genes, search=search, shap=plan))
        variants = ['cat3','cat3+roles','cat3+drop1','cat3+drop2','cat3+drop_redundant',
            'cat3+neighbor_down_106_signed_median_flag','cat3+neighbor_down_106_signed_zero',
            'cat3+interaction1','cat3+interaction2']
        for variant in variants:
            if 'drop_redundant' in variant and plan['deletions']['drop_redundant'] is None:
                continue
            for seed in P['seeds']:
                run_trial(data, OUT, variant=variant, cut=cutoff, seed=seed, profile=P,
                    stage='advanced_features', epoch_policy='control', control_variant='cat3',
                    genes=genes if 'interaction' in variant else None,
                    feature_plan=plan if 'drop' in variant else None)
    display(paired_table(aggregate_runs(prediction_records(OUT, 'advanced_features')), control='cat3'))
else:
    print('Optional deletion and interaction study skipped.')


## 9. Original two-feature graph recipes
Retain the original training-dependent field selection and graph builders for degree, neighbour, two-hop and snapshot feature pairs. This section is separate from the compact fixed-field examples above.


In [ ]:
RUN_ORIGINAL_TREE_PAIRS = globals().get('RUN_ORIGINAL_TREE_PAIRS', False)
if RUN_ORIGINAL_TREE_PAIRS:
    store = SmallGraphFeatureData(DATA_DIR)
    for cutoff in ([21] if PROFILE == 'quick' else [14,21,28]):
        for group in ['raw','degree_pair','neighbor_pair','two_hop_pair','snapshot_pair']:
            for model in FE_TREE_MODELS:
                for seed in P['seeds']:
                    run_original_tree_features(data, OUT, store, group, cutoff, seed, model, P)
    display(paired_table(aggregate_runs(prediction_records(OUT, 'original_graph_pairs'))))
    del store
else:
    print('Optional original graph-pair study skipped.')


## 10. Extended tree-feature search and parameter grids
Keep the source feature generator and declared tree grids as optional experiments. The completion counts in the report describe historical runs; this section records only the configurations actually completed now.


In [ ]:
RUN_EXTENDED_TREE_SEARCH = globals().get('RUN_EXTENDED_TREE_SEARCH', False)
RUN_TREE_GRID = globals().get('RUN_TREE_GRID', False)
MAX_TREE_GRID_CONFIGS = globals().get('MAX_TREE_GRID_CONFIGS', 8)  # None runs every declared configuration.
if RUN_EXTENDED_TREE_SEARCH or RUN_TREE_GRID:
    store = CampaignFeatures(DATA_DIR)
    plan_dir = OUT / 'tree_feature_plan'
    plan_dir.mkdir(parents=True, exist_ok=True)
    plans = store.audit_and_plan(plan_dir)
    save_json(plan_dir / 'feature_specs.json', plans)
    if RUN_EXTENDED_TREE_SEARCH:
        unavailable = []
        for cutoff in [14,21,28]:
            for spec in plans:
                for model in ['xgboost','lightgbm','catboost','hist_gradient_boosting','extra_trees']:
                    if spec['kind'] == 'native_category' and model not in NATIVE_MODELS:
                        continue
                    for seed in P['seeds']:
                        try:
                            run_original_tree_features(data, OUT, store, spec, cutoff, seed, model, P, campaign=True)
                        except FeatureUnavailable as error:
                            unavailable.append(dict(feature=spec['id'],cut=cutoff,model=model,reason=str(error)))
        save_json(plan_dir / 'unavailable_features.json', unavailable)
    if RUN_TREE_GRID:
        raw = next(spec for spec in plans if spec['kind'] == 'raw')
        for model, configs in source_tree_grids().items():
            selected = configs if MAX_TREE_GRID_CONFIGS is None else configs[:MAX_TREE_GRID_CONFIGS]
            for config in selected:
                for cutoff in [14,21]:
                    for seed in P['seeds']:
                        run_original_tree_features(data, OUT, store, raw, cutoff, seed, model, P,
                            campaign=True, overrides=config)
    display(aggregate_runs(prediction_records(OUT, 'original_campaign')))
else:
    print('Optional extended tree search and grids skipped.')


## 11. Scaling, time features and split diagnostics
Compare quantile/robust scaling, removal of time_step and repaired time-sensitive inputs. Random and component splits are diagnostics; they do not replace forward validation.


In [ ]:
RUN_INPUT_DIAGNOSTICS = globals().get('RUN_INPUT_DIAGNOSTICS', False)
if RUN_INPUT_DIAGNOSTICS:
    for kind in ['random','component','time']:
        for variant, scaler in [('raw','quantile'),('raw','robust'),('no_time','quantile')]:
            for seed in P['seeds']:
                run_trial(data, OUT, kind=kind, variant=variant, scaler=scaler, cut=21,
                    seed=seed, profile=P, stage='input_diagnostics')
    for cutoff in [14,21]:
        for variant in ['raw','trend_residual','snapshot_center']:
            for seed in P['seeds']:
                run_trial(data, OUT, variant=variant, cut=cutoff, seed=seed, profile=P,
                    stage='time_proxy_diagnostics', epoch_policy='control')
    display(aggregate_runs(prediction_records(OUT, 'input_diagnostics')))
else:
    print('Optional scaling and split diagnostics skipped.')
